### Pre-process

In [19]:
import html
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                             cohen_kappa_score, classification_report,
                             confusion_matrix)

SEED = 42
VAL_SIZE = 0.10         
SAMPLE_PER_CLASS = 10_000  # از هر کلاس (۱ تا ۵) ۱۰ هزار تا

# پیش پردازش
data = pd.read_csv(
    "sentiment_ready_dataset.csv",
    encoding="utf-8-sig",
    usecols=["text_for_model", "overall"],
    keep_default_na=False,
    low_memory=False,
)
data["overall"] = pd.to_numeric(data["overall"], errors="coerce")
data = data.dropna(subset=["overall"]).copy()
data["overall"] = data["overall"].astype(int)
data = data.rename(columns={"text_for_model": "text"})
data = data[data["text"].str.strip() != ""].reset_index(drop=True)

# نمونه‌گیری: از هر کلاس ۱۰ هزار تا 
data = pd.concat(
    [g.sample(n=min(len(g), SAMPLE_PER_CLASS), random_state=SEED)
     for _, g in data.groupby("overall")]
).reset_index(drop=True)

print("بعد از نمونه‌گیری:", data.shape)
print(data["overall"].value_counts().sort_index())

tr_df, va_df = train_test_split(
    data, test_size=VAL_SIZE, stratify=data["overall"], random_state=SEED
)
tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)

print("train:", tr_df.shape, "- val:", va_df.shape)
print("توزیع train:\n", tr_df["overall"].value_counts(normalize=True).sort_index())
print("توزیع val:\n", va_df["overall"].value_counts(normalize=True).sort_index())

# تابع ارزیابی مشترک 
def evaluate(y_true, y_pred, name="model"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    res = {
        "accuracy":    accuracy_score(y_true, y_pred),
        "f1_micro":    f1_score(y_true, y_pred, average="micro"),
        "f1_macro":    f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "mae":         mean_absolute_error(y_true, y_pred),
        "qwk":         cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }
    print(f"\n===== {name} (validation) =====")
    for k, v in res.items():
        print(f"{k:12s}: {v:.4f}")
    print(classification_report(y_true, y_pred, digits=3))
    print("confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5]))
    return res


test = pd.read_csv("test_data.csv", encoding="utf-8-sig", low_memory=False)
test["text"] = pd.read_csv("sentiment_ready_dataset.csv", encoding="utf-8-sig", low_memory=False)["text_for_model"]
print("test:", test.shape)

بعد از نمونه‌گیری: (50000, 2)
overall
1    10000
2    10000
3    10000
4    10000
5    10000
Name: count, dtype: int64
train: (45000, 2) - val: (5000, 2)
توزیع train:
 overall
1    0.2
2    0.2
3    0.2
4    0.2
5    0.2
Name: proportion, dtype: float64
توزیع val:
 overall
1    0.2
2    0.2
3    0.2
4    0.2
5    0.2
Name: proportion, dtype: float64
test: (20000, 11)


### tf-idf + LogisticRegression

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# تبدیل متن به عدد
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=500_000,
    sublinear_tf=True,
    strip_accents="unicode",
)
Xtr = vectorizer.fit_transform(tr_df["text"])
Xva = vectorizer.transform(va_df["text"])
Xte = vectorizer.transform(test["text"])
print("TF-IDF shape:", Xtr.shape)

# model
clf = LogisticRegression(
    C=4.0,
    solver="saga",
    max_iter=200,
    tol=1e-3,
    class_weight=None,
    random_state=SEED,
)
clf.fit(Xtr, tr_df["overall"])

pred_va_lr = clf.predict(Xva)
f1_lr = f1_score(va_df["overall"], pred_va_lr, average="micro")
print(f"F1 روی validation: {f1_lr:.4f}")

test_pred_lr = clf.predict(Xte)

TF-IDF shape: (45000, 225268)
F1 روی validation: 0.5510


### distilroberta

In [ ]:
import math
import inspect
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          set_seed)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


# تنظیمات مشترک
SEED = 42
MAX_LEN = 256
N_TRAIN = None
VAL_DURING_TRAIN = 1000
EPOCHS = 3
BATCH = 64           
GRAD_ACCUM = 1       
LR = 5e-5

MODEL_NAME = "distilroberta-base"

set_seed(SEED)

# زیرنمونه‌ها 
if N_TRAIN is not None and N_TRAIN < len(tr_df):
    tr_sub, _ = train_test_split(
        tr_df, train_size=N_TRAIN, stratify=tr_df["overall"], random_state=SEED
    )
else:
    tr_sub = tr_df
tr_sub = tr_sub.reset_index(drop=True)
va_small = va_df.sample(n=min(VAL_DURING_TRAIN, len(va_df)), random_state=SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    """توکنایز در لحظه؛ مصرف حافظه‌ی کم برای داده‌ی بزرگ."""
    def __init__(self, texts, tokenizer, labels=None):
        self.texts = list(texts)
        self.tokenizer = tokenizer
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN)
        item = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
        }
        if self.labels is not None:
            item["labels"] = int(self.labels[i]) - 1      
        return item                                       

train_ds = ReviewDataset(tr_sub["text"], tokenizer, tr_sub["overall"])
val_small_ds = ReviewDataset(va_small["text"], tokenizer, va_small["overall"])
val_full_ds = ReviewDataset(va_df["text"], tokenizer)       # بدون لیبل، فقط پیش‌بینی
test_ds = ReviewDataset(test["text"], tokenizer)

sample = train_ds[0]
print(type(sample), list(sample.keys()), "| label:", sample["labels"])
assert isinstance(sample, dict) and "input_ids" in sample

#  مدل کامل با fine-tune (بدون freeze)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"f1_micro": f1_score(labels, preds, average="micro")}


total_steps = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM)) * EPOCHS
warmup_steps = int(total_steps * 0.06)
print(f"total steps: {total_steps} | warmup: {warmup_steps}")

args_dict = dict(
    output_dir="distilroberta_out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    disable_tqdm=False,          # نوار پیشرفت آموزش و ارزیابی
    logging_steps=50,            
    report_to="none",
    seed=SEED,
)

valid = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
dropped = [k for k in args_dict if k not in valid]
args = TrainingArguments(**{k: v for k, v in args_dict.items() if k in valid})

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_small_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

# ارزیابی نهایی روی کل ولیدیشن 
out_va = trainer.predict(val_full_ds)
pred_va = out_va.predictions.argmax(axis=-1) + 1
f1_va = f1_score(va_df["overall"], pred_va, average="micro")
print(f"F1 (micro) روی validation: {f1_va:.4f}")


<class 'dict'> ['input_ids', 'attention_mask', 'labels'] | label: 2


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


total steps: 2112 | warmup: 126


Epoch,Training Loss,Validation Loss,F1 Micro
1,0.908006,0.852913,0.636000
2,0.772220,0.832179,0.639000
3,0.645894,0.867405,0.636000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

F1 (micro) روی validation: 0.6400


1     1923
2     1354
3     2048
4     4463
5    10212
Name: count, dtype: int64


In [20]:
# پیش‌بینی روی test
out_te = trainer.predict(test_ds)
test_pred = out_te.predictions.argmax(axis=-1) + 1      # 0..4 -> 1..5

assert test_pred.min() >= 1 and test_pred.max() <= 5, "برچسب‌ها خارج از ۱ تا ۵ هستند"
assert len(test_pred) == len(test), "تعداد پیش‌بینی‌ها با تعداد نمونه‌های test یکی نیست"

# ساخت فایل ارسالی 
submission = pd.DataFrame({"predicted": test_pred})
submission.to_csv("q2_submission.csv", index=False)

sub = pd.read_csv("q2_submission.csv")

# چک قالب فایل
assert list(sub.columns) == ["predicted"], f"ستون‌ها: {list(sub.columns)}"
assert len(sub) == len(test), f"تعداد ردیف‌ها {len(sub)} ولی test {len(test)} است"
assert pd.api.types.is_integer_dtype(sub["predicted"]), "مقادیر باید عدد صحیح باشند"
assert sub["predicted"].between(1, 5).all(), "مقدار خارج از ۱ تا ۵ وجود دارد"

# ۱۰ نمونه‌ی تصادفی: شماره‌ی ردیف | امتیاز پیش‌بینی‌شده | متن
print("\n 5 نمونه‌ی تصادفی:")
sample = sub.sample(min(5, len(sub)), random_state=SEED).sort_index()
for i, row in sample.iterrows():
    text = " ".join(str(test["text"].iloc[i]).split())
    print(f"predicted={row['predicted']} - {text[:150]}")


 5 نمونه‌ی تصادفی:
predicted=3 - Decent for casual listening but not consistently good for classical music. I bought these on the recommendation of a reviewer who purchased an MP3 pla
predicted=1 - Not for professional use. Ghosting, lens flair, light abrasions ... I would not recommend purchasing this filter for anything other than an lens prote
predicted=5 - Great Switch! Great Price! Great Performance!. Took my time waiting to write this review because Netgear had some firmware issues with it that preclud
predicted=5 - they are mostly because of an issue or poor performance. However. I rarely write reviews and when I do, they are mostly because of an issue or poor pe
predicted=5 - Effective and simple. Great and simple device. One of the few simple door alarms that also has the chime feature. The chime feature is the reason I bo
